# Digital Ocean cloud computing


## Overview of steps
- Setup ( one time )
  - Create a [DO account]( https://cloud.digitalocean.com/registrations/new )
  - Create a public/private SSH key
  - Add the key to your DO account
- Each time
  - Launch an instance configured with the public SSH key
  - SSH into the instance with or without a tunnel
- even better
  - Store a JSON object with the options
  - Run a curl command using the JSON to launch instances


Cloud Functions
- https://cloud.digitalocean.com/functions


ABQ Bus data
- https://data.cabq.gov/transit/realtime/route/allroutes.json

## Launch, query, destory instances via REST API




[API Reference](https://docs.digitalocean.com/reference/api/api-reference/)

### Setup


Create an Personal Access Token ( PAT ) and store it in your Colab vault, naming it `digital_ocean_api`.

[DO Tokens](https://cloud.digitalocean.com/account/api/tokens)


In [ ]:
import os
import time
from google.colab import userdata
import requests
import yaml


In [ ]:
digital_ocean_token = userdata.get('digital_ocean_api')

In [ ]:
f"{digital_ocean_token[:10]}...{digital_ocean_token[-5:]}"

In [ ]:
# a bogus token for testing; uncomment if needed
# digital_ocean_token = "foobar"

To find your VPC uuid, go here: https://cloud.digitalocean.com/networking/vpc

Then click on the region where you are going to start your droplets.  The VPC uuid will be in the **address bar** ( i.e. in the URL. )

In [ ]:
ssh_fingerprint = "ee:6e:ed:eb:ed:70:1c:66:db:0c:07:21:61:77:df:42"
ssh_fingerprint


In [ ]:
vpc_uuid = "981e091e-2d0b-4773-8354-966a14c68412"
vpc_uuid


In [ ]:
timestamp = str(int(time.time()))
timestamp

```curl -X POST -H 'Content-Type: application/json' \
    -H 'Authorization: Bearer '$TOKEN'' \
    -d '{"name":"ubuntu-s-1vcpu-512mb-10gb-sfo3-01",
        "size":"s-1vcpu-512mb-10gb",
        "region":"sfo3",
        "image":"ubuntu-24-04-x64",
        "vpc_uuid":"981e091e-2d0b-4773-8354-966a14c68412"}' \
    "https://api.digitalocean.com/v2/droplets"
  ```


### Launching an instance

```python
url = "https://api.digitalocean.com/v2/droplets"
url

```

In [ ]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {digital_ocean_token}"
}
headers | { "Authorization": f"Bearer ... {digital_ocean_token[-5:]}" }


In [ ]:
data = {
    "name": f"ubuntu-temp-via-curl-{timestamp}",
    "image": "ubuntu-24-04-x64",
    "size": "s-2vcpu-4gb",
    "region": "sfo3",
    "ssh_keys": [
        ssh_fingerprint
    ],
    "vpc_uuid": vpc_uuid,
}
data


In [ ]:
response = requests.post(url, headers=headers, json=data)
response


In [ ]:
droplet = response.json()
droplet.keys()


In [ ]:
droplet['links']


### Get Droplet info, e.g. IP address


The IP address should be in the response dictionary, but it isn't.  That's because the VM doesn't have its networking configured before the response is sent.


In [ ]:
droplet['droplet']['networks']


Wait a bit, then request information about the droplet using its ID.


In [ ]:
# extract droplet id
id = droplet['droplet']['id']
id

In [ ]:
# droplet id endpoint
id_url = f"{url}/{id}"
id_url

In [ ]:
# GET request using ID endpoint and same headers, but no payload
response = requests.get(id_url, headers=headers)
response


In [ ]:
# extract droplet id
droplet_dict = response.json()['droplet']
droplet_dict['id']


In [ ]:
# get public IP address
ips = [
  x['ip_address']
    for x in droplet_dict['networks']['v4']
      if x['type'] == 'public'
]
ip = ips[0]
ip

The `droplet_dict` variable has much information about the droplet in a dictionary format.

In [ ]:
print(yaml.dump(droplet_dict))


### Delete droplet

This is the same url as the get, except the method is delete.

In [ ]:
# verify url
id_url


In [ ]:
# verify headers
headers | { "Authorization": f"Bearer ... {digital_ocean_token[-5:]}" }


In [ ]:
response = requests.delete( id_url, headers=headers )
response


In [ ]:
response.status_code, response.reason

In [ ]:
response.headers

In [ ]:
# @title SSH into the instance
print(f"ssh -L 5150:localhost:5150 root@{ip}")


### Configuring an instance

In [ ]:
# @title Upgrade and reboot
%%markdown
```bash
  {
sleep 10
export DEBIAN_FRONTEND=noninteractive
apt-get update
apt-get -y dist-upgrade
}
{ sleep 5 ; reboot ; } & exit
```

In [ ]:
# @title Install Docker
%%markdown
```bash
source /etc/os-release
export DEBIAN_FRONTEND=noninteractive
apt-get update
apt-get install -y apt-transport-https ca-certificates curl software-properties-common
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | apt-key add -
yes | add-apt-repository "deb [arch=amd64] https://download.docker.com/linux/ubuntu ${UBUNTU_CODENAME} stable"
apt-get update
apt-cache policy docker-ce
apt-get install -y docker-ce
systemctl status docker
```

### Run Jupyter Lab

See [Jupyter Lab in Docker](https://github.com/rwcitek/Docker.jupyter-lab#launch-container)



In [ ]:
# @title Jupyter light
%%markdown
```bash
host=$( curl -s 'https://api.ipify.org?format=json' | jq -r .ip )
SHARED=/tmp/jupyter
mkdir -p "${SHARED}"
docker \
    run \
    --rm \
    -d \
    -p :5150:8888 \
    -e JUPYTER_ENABLE_LAB=yes \
    -v "${SHARED}":/home/jovyan/shared \
    -w /home/jovyan/shared \
    --name jupyter-lab \
    rwcitek/jupyter.light \
      jupyter lab --allow-root --ip=0.0.0.0

while true; do
  token=$( docker container logs --since 5s jupyter-lab 2>&1 | grep -m1 -o token=.* )
  [ "${token}" ] && echo -e "\n\n\nhttp://${host}:5150/lab?${token}\n\n\n" && break
  sleep 2
done
```


## Creating a serverless function


Here's a link to creating a Docker container to enable creating a serverless function in Digital Ocean.

[DO Serverless Function](https://gist.github.com/rwcitek/1a03bfd7b59595f4e1e36f8dc22cf393#file-digital-ocean-function-docker-md)

## Cloud Function

In [ ]:
import requests


In [ ]:
from google.colab import userdata
do_token = userdata.get('do_function')

In [ ]:
!curl -X POST -H "Content-Type: application/json" -H "Authorization: Basic {do_token}" "https://faas-nyc1-2ef2e6cc.doserverless.co/api/v1/namespaces/fn-20092b5b-b1bc-44d3-85c8-474eaa7ac80c/actions/sample/hello?name=Robert&blocking=true&result=true"



In [ ]:
!curl -s 'https://faas-nyc1-2ef2e6cc.doserverless.co/api/v1/web/fn-20092b5b-b1bc-44d3-85c8-474eaa7ac80c/default/add_two_numbers?num1=1&num2=3'

In [ ]:
!curl -s "https://faas-nyc1-2ef2e6cc.doserverless.co/api/v1/web/fn-20092b5b-b1bc-44d3-85c8-474eaa7ac80c/sample/hello?name=Robert"

## Example cloud function



This pulls an image from Yosemite and saves to cloud object storage.


In [ ]:

import os
from datetime import datetime, timezone
import boto3
import requests

# fetch env settings
WEBCAM_URL = os.environ.get('WEBCAM_URL', 'https://cdn.pixelcaster.com/public.pixelcaster.com/snapshots/yosemite-sentinel/latest.jpg')
DO_SPACES_NAME = os.environ.get('DO_SPACES_NAME', 'your-webcam-images')
DO_SPACES_REGION = os.environ.get('DO_SPACES_REGION', 'nyc3')
DO_SPACES_KEY = os.environ.get('DO_SPACES_KEY')
DO_SPACES_SECRET = os.environ.get('DO_SPACES_SECRET')

def main(args):
  # Create timestamp
  now_utc = datetime.now(timezone.utc).isoformat(timespec='seconds').replace("+00:00", "Z")

  # Verify envs
  envs = [ f"{key}={value[:10]}" for key, value in os.environ.items() if key.startswith("DO_")]
  envs = "\n".join(envs)

  # create boto connection
  session = boto3.session.Session()
  s3_client = session.client(
    's3',
    region_name=DO_SPACES_REGION,
    endpoint_url=f'https://{DO_SPACES_REGION}.digitaloceanspaces.com',
    aws_access_key_id=DO_SPACES_KEY,
    aws_secret_access_key=DO_SPACES_SECRET
  )

  # fetch image
  response = requests.get(WEBCAM_URL, stream=True)

  # create object name
  file_extension = "jpg"
  spaces_object_name = f"webcam_images/{now_utc}.{file_extension}"

  # save image to spaces
  s3_client.upload_fileobj(
    response.raw,
    DO_SPACES_NAME,
    spaces_object_name,
    ExtraArgs={
      'ContentType': response.headers['Content-Type'],
      'ACL': 'public-read',
    }

  )

  # generate output
  greeting = f'''
    Now: {now_utc}
    Region: {DO_SPACES_REGION}
    output_name: {spaces_object_name}
    {envs}

  '''
  print(greeting)
  return {"body": greeting}

